# calibrated-typed-decisions: update the card's results section
Replaces the single-run results with the multi-seed and ensemble results. Card only, no data changes. Run once with `PUBLISH = False` first.

## Cell 1 — Install, Drive and Hugging Face login
No GPU needed. The login needs a WRITE token.

In [ ]:
!pip install -q datasets huggingface_hub
from google.colab import drive
drive.mount("/content/drive")
from huggingface_hub import notebook_login
notebook_login()

## Cell 2 — Build the new results section
Reads the files Steps 2 and 3 saved, and writes the section. Nothing is uploaded yet.

In [4]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import DatasetCard

PUBLISH  = True        # True: upload the updated card (data is not touched)
HF_REPO  = "m-lagnajit/calibrated-typed-decisions"
BASE_DIR = "/content/drive/MyDrive/ctd_student_v2"
RUNS     = ["A", "B", "C", "D", "E"]
TARGETS  = {"A": "one-hot `gold`", "B": "`gold`, label-smoothed (0.1)", "C": "`soft_label`",
            "D": "`0.5 * one_hot(gold) + 0.5 * soft_label`", "E": "`corrected_soft_label`"}

per_seed = pd.read_csv(f"{BASE_DIR}/step2_per_seed.csv")
cons     = pd.read_csv(f"{BASE_DIR}/step2_consistency.csv")
ens      = pd.read_csv(f"{BASE_DIR}/step3_ensembles.csv")
ens_boot = pd.read_csv(f"{BASE_DIR}/step3_ensemble_bootstrap.csv")
n_seeds  = per_seed["seed"].nunique()
assert n_seeds >= 3 and set(per_seed["run"]) == set(RUNS), "expected all five runs with three seeds"

def md_table(header, rows):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    return "\n".join(out + ["| " + " | ".join(map(str, r)) + " |" for r in rows])

# Single students: mean (lowest–highest) over seeds
def mr(split, run, m):
    x = per_seed[(per_seed.split == split) & (per_seed.run == run)][m]
    return f"{x.mean():.3f} ({x.min():.3f}–{x.max():.3f})"
seed_table = md_table(
    ["run", "test accuracy", "test ECE", "test log loss", "held-out accuracy", "held-out ECE", "held-out log loss"],
    [[r] + [mr(s, r, m) for s in ("test", "heldout") for m in ("accuracy", "ECE", "log loss")] for r in RUNS])

# Ensembles
def ev(split, run, m):
    return f"{ens[(ens.split == split) & (ens.run == run)][f'{m}: ensemble'].iloc[0]:.3f}"
ens_table = md_table(
    ["run", "test accuracy", "test ECE", "test log loss", "held-out accuracy", "held-out ECE", "held-out log loss"],
    [[r] + [ev(s, r, m) for s in ("test", "heldout") for m in ("accuracy", "ECE", "log loss")] for r in RUNS])

# What held up under BOTH checks: every seed (Step 2) AND the ensemble bootstrap (Step 3)
def direction(verdict):
    return "better" if "better" in verdict else ("worse" if "worse" in verdict else None)
eb = ens_boot.assign(comparison=ens_boot["comparison"].str.replace(" (ensembles)", "", regex=False))
both = cons.merge(eb, on=["split", "comparison", "measure"], suffixes=("_seeds", "_ens"))
both["dir_seeds"] = both["verdict_seeds"].map(direction)
both["dir_ens"] = both["verdict_ens"].map(direction)
robust = both[both["dir_seeds"].notna() & (both["dir_seeds"] == both["dir_ens"])]
def robust_lines(split):
    lines = []
    for _, r in robust[robust.split == split].iterrows():
        cand, base = r["comparison"].split(" vs ")
        lines.append(f"- {cand} {r['dir_seeds']} than {base} on {r['measure']}")
    return "\n".join(lines) or "- (none)"

# Held-out baselines and the size of training luck, computed from the data itself
gold_h = pd.Series(load_dataset(HF_REPO, split="heldout")["gold"])
freq = gold_h.value_counts(normalize=True)
majority, base_ll = float(freq.iloc[0]), float(-(freq * np.log(freq)).sum())
luck = (per_seed[per_seed.split == "heldout"].groupby("run")["accuracy"].agg(lambda x: x.max() - x.min()))
max_luck_run, max_luck = luck.idxmax(), float(luck.max())

SECTION = f"""## Which training target to use

We trained ModernBERT-base students on the train split with five different targets, each with
**{n_seeds} random seeds** (3 epochs; best epoch chosen on validation log loss). Every student then gets
**2 temperature dials** fitted on validation: one for choice questions, and one plus a yes/no shift for
yes/no questions. We compared this with one dial per source and question type, and with no dial, by
cross-fitting on validation: 2 dials gave the lowest log loss for all five targets.

{md_table(["run", "target"], [[r, TARGETS[r]] for r in RUNS])}

### Single students: mean over {n_seeds} seeds (lowest–highest)

{seed_table}

### Ensembles: each run's {n_seeds} calibrated students averaged

{ens_table}

On held-out, always answering "{freq.index[0]}" scores {majority:.3f} accuracy, and always predicting
the label frequencies gives a log loss of {base_ll:.3f}.

### What held up under both checks

Listed only if it held in **every seed** *and* in a paired bootstrap over questions for the ensembles.

**Test split** (the six training sources):
{robust_lines("test")}

**Held-out PubMedQA** (never trained on):
{robust_lines("heldout")}

### In short

- **On data like the six training sources, one-hot `gold` with a dial is as good as any soft target.**
  `soft_label` alone (C) is the weakest there, because it inherits the teacher's mistakes.
- **On the held-out domain, targets that keep the teacher's opinion (C, D, E) give more honest
  confidence** than targets built from `gold` alone (A, B), but not higher accuracy. Every student stays
  close to the baselines above, so transfer to this domain is weak.
- **The most balanced choice in our runs was D's ensemble**: never significantly worse than the best
  target on either split.
- **Judge targets on several seeds.** Retraining alone moved one run's held-out accuracy by
  {max_luck:.3f} (run {max_luck_run}).

"""

card = DatasetCard.load(HF_REPO)
text = card.text
print("headings on the current card:")
for h in [l for l in text.splitlines() if l.startswith("## ")]:
    print("  ", h)

START = "## Which training target to use"
if START in text:
    # Replace the old section, up to the next top-level heading (whatever it's called)
    start = text.index(START)
    nxt = re.search(r"\n## ", text[start + len(START):])
    end = start + len(START) + nxt.start() + 1 if nxt else len(text)
    NEW_TEXT = text[:start] + SECTION + text[end:]
    print("\n-> replacing the existing results section")
else:
    # No results section yet: put it before the first of these headings that exists
    anchor = next((h for h in ["## Teacher calibration on the test split", "## Sources", "## Limitations"]
                   if h in text), None)
    if anchor:
        i = text.index(anchor)
        NEW_TEXT = text[:i] + SECTION + text[i:]
        print(f"\n-> no results section yet: inserting it before '{anchor}'")
    else:
        NEW_TEXT = text.rstrip() + "\n\n" + SECTION
        print("\n-> no results section yet: adding it at the end")
assert NEW_TEXT.count(START) == 1, "the card would end up with two results sections; nothing changed"
print("\n" + SECTION)
print(f"(card: {len(text)} -> {len(NEW_TEXT)} characters; only this section changes)")

headings on the current card:
   ## What changed in v2.1
   ## What changed from v1
   ## Splits
   ## Fields
   ## Teacher calibration on the test split
   ## Sources
   ## Limitations

-> no results section yet: inserting it before '## Teacher calibration on the test split'

## Which training target to use

We trained ModernBERT-base students on the train split with five different targets, each with
**3 random seeds** (3 epochs; best epoch chosen on validation log loss). Every student then gets
**2 temperature dials** fitted on validation: one for choice questions, and one plus a yes/no shift for
yes/no questions. We compared this with one dial per source and question type, and with no dial, by
cross-fitting on validation: 2 dials gave the lowest log loss for all five targets.

| run | target |
|---|---|
| A | one-hot `gold` |
| B | `gold`, label-smoothed (0.1) |
| C | `soft_label` |
| D | `0.5 * one_hot(gold) + 0.5 * soft_label` |
| E | `corrected_soft_label` |

### Single students:

## Cell 3 — Publish the card
Uploads only the card. The data stays exactly as v2.1.

In [5]:
if not PUBLISH:
    print("Dry run: nothing uploaded. Read the section above, then set PUBLISH = True in Cell 2 and rerun Cells 2 and 3.")
else:
    card.text = NEW_TEXT
    card.push_to_hub(HF_REPO, commit_message="Card: multi-seed and ensemble results for training targets")
    print(f"Card updated: https://huggingface.co/datasets/{HF_REPO}")

Card updated: https://huggingface.co/datasets/m-lagnajit/calibrated-typed-decisions
